# Role Assignment Coherence Judge

This notebook compares two role-assignment artifacts with a language-model-as-a-judge workflow.
It relies on `RoleAssignmentJudge` (OPENAI judge by default) that was implemented in `src/role_evaluator`.

## 1. Configuration
- Set a gpt API key (best judged model) via the `OPENAI_API_KEY` environment variable.
- Update the file paths below to point at the two role-assignment JSON dumps you want to compare.

In [1]:
import os
import sys
from pathlib import Path
import json
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent.parent  # Go up two levels to reach project root
sys.path.append(str(PROJECT_ROOT))  # Add project root to Python path

from src.role_evaluator import RoleAssignmentJudge, DatasetComparisonSpec
SPIDER_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_spider_dev_20251005_145223.json'
SPIDER_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_spider_dev_20250930_230436.json'
BIRD_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_bird_dev_20251005_164218.json'
BIRD_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_bird_dev_selected.json'
LIVESQL_ASSIGNMENT_NAIVE = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_20251005_164446_livesql_dev.json'
LIVESQL_ASSIGNMENT_WITH_COT = PROJECT_ROOT / 'outputs/laaj_role/role_assignments_20251001_142136_livesql_dev.json'

# Read gpt API key from env (edit this cell to hardcode if preferred)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '').strip()
if not OPENAI_API_KEY:
    raise ValueError('Set OPENAI_API_KEY in your environment or assign OPENAI_API_KEY manually.')

DATASET_COMPARISONS = [
    DatasetComparisonSpec(
        label='spider',
        assignment_a_path=SPIDER_ASSIGNMENT_NAIVE,
        assignment_b_path=SPIDER_ASSIGNMENT_WITH_COT,
    ),
    DatasetComparisonSpec(
        label='bird',
        assignment_a_path=BIRD_ASSIGNMENT_NAIVE,
        assignment_b_path=BIRD_ASSIGNMENT_WITH_COT,
    ),
    DatasetComparisonSpec(
        label='livesql',
        assignment_a_path=LIVESQL_ASSIGNMENT_NAIVE,
        assignment_b_path=LIVESQL_ASSIGNMENT_WITH_COT,
    ),
]

/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Instantiate the judge
GPT-5 is used as the evaluator to ensure a different model from the role generator.

In [2]:
JUDGE_MODEL = 'gpt-4o'  # Top gpt model as of writing
judge = RoleAssignmentJudge(
    model=JUDGE_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0.9,
    top_p=0.9,
    max_completion_tokens=800,
)
judge


## 3. Run evaluation
This triggers one OPENAI call per overlapping database in the two artifacts. Optional filters:
- `databases=[...]`: restrict to a selected list.
- `limit=N`: judge only the first *N* shared databases (useful for pilots).

In [ ]:
batch_result = judge.evaluate_many(
    DATASET_COMPARISONS,
    combined_progress=True,
    dataset_progress=True,
    progress_description='Evaluating all databases',
)
batch_result.summary.to_dict()

Evaluating all databases: 100%|██████████| 44/44 [01:22<00:00,  1.87s/db]


{'wins_a': 16,
 'wins_b': 28,
 'ties': 0,
 'unknown': 0,
 'comparisons': 44,
 'decisive_comparisons': 44,
 'win_rate_a': 0.36363636363636365,
 'win_rate_b': 0.6363636363636364}

In [4]:
per_dataset_summary = {
    label: per_result.summary.to_dict()
    for label, per_result in batch_result.results.items()
}
per_dataset_summary

{'spider': {'wins_a': 5,
  'wins_b': 10,
  'ties': 0,
  'unknown': 0,
  'comparisons': 15,
  'decisive_comparisons': 15,
  'win_rate_a': 0.3333333333333333,
  'win_rate_b': 0.6666666666666666},
 'bird': {'wins_a': 3,
  'wins_b': 8,
  'ties': 0,
  'unknown': 0,
  'comparisons': 11,
  'decisive_comparisons': 11,
  'win_rate_a': 0.2727272727272727,
  'win_rate_b': 0.7272727272727273},
 'livesql': {'wins_a': 8,
  'wins_b': 10,
  'ties': 0,
  'unknown': 0,
  'comparisons': 18,
  'decisive_comparisons': 18,
  'win_rate_a': 0.4444444444444444,
  'win_rate_b': 0.5555555555555556}}

In [5]:
result_all = {
    'overall': batch_result.summary.to_dict(),
    'per_dataset': per_dataset_summary,
}
result_all

{'overall': {'wins_a': 16,
  'wins_b': 28,
  'ties': 0,
  'unknown': 0,
  'comparisons': 44,
  'decisive_comparisons': 44,
  'win_rate_a': 0.36363636363636365,
  'win_rate_b': 0.6363636363636364},
 'per_dataset': {'spider': {'wins_a': 5,
   'wins_b': 10,
   'ties': 0,
   'unknown': 0,
   'comparisons': 15,
   'decisive_comparisons': 15,
   'win_rate_a': 0.3333333333333333,
   'win_rate_b': 0.6666666666666666},
  'bird': {'wins_a': 3,
   'wins_b': 8,
   'ties': 0,
   'unknown': 0,
   'comparisons': 11,
   'decisive_comparisons': 11,
   'win_rate_a': 0.2727272727272727,
   'win_rate_b': 0.7272727272727273},
  'livesql': {'wins_a': 8,
   'wins_b': 10,
   'ties': 0,
   'unknown': 0,
   'comparisons': 18,
   'decisive_comparisons': 18,
   'win_rate_a': 0.4444444444444444,
   'win_rate_b': 0.5555555555555556}}}

## 4. Inspect detailed decisions
Each row corresponds to one database comparison.

In [6]:
import pandas as pd

decision_records = []
for label, per_result in batch_result.results.items():
    for decision in per_result.decisions:
        record = decision.to_dict()
        record['dataset'] = label
        decision_records.append(record)

decisions_df = pd.DataFrame(decision_records)
decisions_df

,database,winner,rationale,confidence,raw_answer,metadata,dataset
0,battle_death,B,Option B offers clearer role naming consistenc...,high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
1,concert_singer,B,"Option B provides clear, non-overlapping roles...",high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
2,course_teach,A,Option A offers a clear separation of responsi...,high,Winner: A \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
3,cre_Doc_Template_Mgt,A,Option A provides a clearer separation of duti...,high,Winner: A \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
4,dog_kennels,A,Option A provides clear role definitions with ...,medium,Winner: A \nConfidence: medium \nRationale: ...,{'query': 'Compare two RBAC role proposals for...,spider
5,employee_hire_evaluation,A,Option A provides a more coherent and comprehe...,high,Winner: A \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
6,museum_visit,B,Option B eliminates redundancy by combining mu...,high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
7,network_1,B,Option B provides a coherent and realistic rol...,high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
8,orchestra,B,Option B offers a more comprehensive and clear...,high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider
9,pets_1,B,Option B provides distinct roles without overl...,high,Winner: B \nConfidence: high \nRationale: Op...,{'query': 'Compare two RBAC role proposals for...,spider


## 5. Persist the raw outputs (optional)
Use this to archive the JSON that records both per-db judgements and aggregate statistics.

In [7]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = Path('outputs/judge_results') / f'coherence_judge_{timestamp}.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open('w', encoding='utf-8') as handle:
    json.dump(batch_result.to_dict(), handle, indent=2)
output_path

PosixPath('outputs/judge_results/coherence_judge_20251009_162232.json')